# DS bid–ask option pricing: demo

Walkthrough of the model from the thesis (Dempster–Shafer multiplicative random walk): 1) parameters, 2) European bid/ask and the CRR limit, 3) theoretical spreads, 4) bid/ask trees, 5) American put, 6) calibration to META options.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, matplotlib.pyplot as plt
from dsbidask import *
from dsbidask.pricing import european_tree
from dsbidask.theory import spread_vs_epsilon
from dsbidask.data import load_quotes
from dsbidask.calibration import sse, calibrate
from dsbidask import plotting as P

## 1. Parameters

The stock moves by factor `u` or `d` each period. `b_u = (R-d)/(u-d)` and `b_d` are the lower masses of the up and down moves; `1 - b_u - b_d >= 0` measures ambiguity. Zero ambiguity is the classical CRR model.

In [ ]:
S0, K, T = 100, 100, 5
u, d, R = 1.2, 0.8, 1.04
bu = b_up(u, d, R)
b_d = 0.4 * 0.999
print(f'b_u = {bu:.4f}, b_d = {b_d:.4f}, ambiguity = {1 - bu - b_d:.4f}')

## 2. European bid/ask prices

For monotone payoffs the bid/ask are binomial sums with different weights (`dsbidask/pricing.py`).

In [ ]:
print('call bid/ask:', european_call_bid_ask(S0, K, u, d, b_d, R, T))
print('put  bid/ask:', european_put_bid_ask(S0, K, u, d, b_d, R, T))

# Zero ambiguity: both collapse to the CRR price
print('call at b_d = 1-b_u:', european_call_bid_ask(S0, K, u, d, 1 - bu, R, T))
print('CRR call           :', crr_price(lambda s: np.maximum(s - K, 0), S0, u, d, R, T))

## 3. Theoretical spread vs ε  (b_d = ε(1 − b_u))

In [ ]:
curves = {f'T={t}': spread_vs_epsilon('call', 100, 100, 1.05, 0.95, 1.01, t) for t in (5, 10, 15, 20)}
P.plot_spread_vs_epsilon(curves, 'Ask - bid', 'European call: theoretical spread');

## 4. Bid and ask trees

In [ ]:
S, bid, ask = european_tree(lambda s: np.maximum(s - K, 0), S0, u, d, b_d, R, T, 'increasing')
P.plot_tree(S, 'Stock tree'); P.plot_tree(bid, 'European call: bid tree'); P.plot_tree(ask, 'European call: ask tree');

## 5. American put

Backward induction with early exercise; American bid/ask dominate the European ones.

In [ ]:
ab, aa = american_put_bid_ask(S0, K, u, d, b_d, R, T)
eb, ea = european_put_bid_ask(S0, K, u, d, b_d, R, T)
print(f'American put bid/ask: {ab:.4f} / {aa:.4f}')
print(f'European put bid/ask: {eb:.4f} / {ea:.4f}')

## 6. Calibration to META options (2023-09-29 → 2023-10-27)

The thesis parameters (free `d`, PSO seed 12345) are evaluated below. Re-running the optimiser takes several minutes: set `RERUN = True`.

In [ ]:
q = load_quotes()
print(len(q.K_call), 'calls,', len(q.K_put), 'puts')
params = dict(u=1.02354592, d=0.96425885, b_d=0.39260853)
params['b_u'] = b_up(params['u'], params['d'], R_DEFAULT)
RERUN = False
if RERUN:
    params = calibrate(q, free_d=True)
print({k: round(float(v), 6) for k, v in params.items()})
print('SSE =', round(sse(params['u'], params['d'], params['b_d'], q), 4))

In [ ]:
P.plot_calibration_fit(q, params, R_DEFAULT, T_DEFAULT);

## Limitations

- The closed-form bid/ask require **monotone** payoffs; a straddle (call + put) is not covered.
- The model-implied spread is narrower than the market spread: the thesis fits the shape of the curves, not their level.